In [132]:
import pandas as pd
import re
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
from collections import Counter

In [134]:
DATASETS = ""
SAVE_PATH = ""

In [135]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
train_df = pd.read_csv(f"{DATASETS}/train_preprocess.tsv", sep='\t', header=None, names=['text', 'label'])
val_df = pd.read_csv(f"{DATASETS}/valid_preprocess.tsv", sep='\t', header=None, names=['text', 'label'])
test_df = pd.read_csv(f"{DATASETS}/test_preprocess.tsv", sep='\t', header=None, names=['text', 'label'])

label2idx = {'positive': 0, 'neutral': 1, 'negative': 2}
train_df['label'] = train_df['label'].map(label2idx)
val_df['label'] = val_df['label'].map(label2idx)
test_df['label'] = test_df['label'].map(label2idx)

print(f"Training: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Testing: {len(test_df)}")

In [137]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['text_clean'] = train_df['text'].apply(clean_text)
val_df['text_clean'] = val_df['text'].apply(clean_text)
test_df['text_clean'] = test_df['text'].apply(clean_text)

In [138]:
VOCAB_SIZE = 20000
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-3
NUM_LABELS = 3

In [139]:
all_words = ' '.join(train_df['text_clean'].values).split()
word_freq = Counter(all_words)
most_common = word_freq.most_common(VOCAB_SIZE - 2)
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, _ in most_common:
    word2idx[word] = len(word2idx)

idx2word = {v: k for k, v in word2idx.items()}
actual_vocab_size = len(word2idx)

In [140]:
def tokenize_text(text, word2idx, max_len):
    tokens = text.split()[:max_len]
    indices = [word2idx.get(w, 1) for w in tokens]
    indices = indices + [0] * (max_len - len(indices))
    return indices

In [141]:
class SmSADataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
        self.data = [
            (
                torch.tensor(tokenize_text(text, word2idx, max_len), dtype=torch.long),
                torch.tensor(label, dtype=torch.long)
            )
            for text, label in zip(texts, labels)
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_ids, label = self.data[idx]
        return {'input_ids': input_ids, 'label': label}

In [ ]:
train_dataset = SmSADataset(train_df['text_clean'].values, train_df['label'].values, word2idx, MAX_LEN)
val_dataset = SmSADataset(val_df['text_clean'].values, val_df['label'].values, word2idx, MAX_LEN)
test_dataset = SmSADataset(test_df['text_clean'].values, test_df['label'].values, word2idx, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [142]:
class SelfAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SelfAttention, self).__init__()
        self.attention_fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_output):
        scores = self.attention_fc(lstm_output)
        attention_weights = torch.softmax(scores, dim=1)
        context = torch.sum(attention_weights * lstm_output, dim=1)
        attention_weights = attention_weights.squeeze(-1)
        return context, attention_weights

In [143]:
class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout):
        super(BiLSTMAttentionClassifier, self).__init__()
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embed_dim,
            padding_idx = 0
        )
        self.embed_dropout = nn.Dropout(p=dropout)

        self.lstm = nn.LSTM(
            input_size = embed_dim,
            hidden_size = hidden_dim,
            num_layers = num_layers,
            bidirectional = True,
            batch_first = True,
            dropout = dropout if num_layers > 1 else 0
        )

        self.attention = SelfAttention(hidden_dim)
        self.dropout = nn.Dropout(p=dropout)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, input_ids):
        embedded = self.embed_dropout(self.embedding(input_ids))
        lstm_out, _ = self.lstm(embedded)
        context, attention_weights = self.attention(lstm_out)
        out = self.dropout(context)
        out = self.relu(self.fc1(out))
        logits = self.fc2(out)
        return logits, attention_weights

In [ ]:
model = BiLSTMAttentionClassifier(
    vocab_size = actual_vocab_size,
    embed_dim = EMBED_DIM,
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    num_classes = NUM_LABELS,
    dropout = DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameter : {total_params:,}")
print(f"Parameter trainable: {trainable_params:,}")
print(f"Struktur Model: ")
print(model)

In [145]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = LR,
    weight_decay = 0.01
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode = 'min',
    patience = 3,
    factor = 0.5,
)

In [146]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits, _ = model(input_ids)

        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            logits, _ = model(input_ids)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 5

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc  = val_acc
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f"{SAVE_PATH}/best_bilstm_attention.pt")
    else:
        patience_counter += 1

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print("EARLY STOPPING!")
        break

print(f"Best Val Accuracy : {best_val_acc:.4f}")
print(f"Best Val Loss : {best_val_loss:.4f}")

In [ ]:
model.load_state_dict(torch.load(f"{SAVE_PATH}/best_bilstm_attention.pt"))
_, test_acc, test_preds, test_true = evaluate(model, test_loader, criterion)
print(classification_report(
    test_true, test_preds,
    target_names=['positive', 'neutral', 'negative'],
    digits=4
))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
ax.plot(history['val_acc'], 'r-s', label='Val Acc', linewidth=2)
ax.set_title('BiLSTM + Attention - Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
ax.plot(history['val_loss'], 'r-s', label='Val Loss', linewidth=2)
ax.set_title('BiLSTM + Attention - Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.show()